# 🎃 ZapalloAI — Notebook 03: Entrenamiento YOLOv11n-cls

**Universidad de las Fuerzas Armadas ESPE**  
Estudiantes: César Loor, Camilo Orrico  
Docente: Ing. Doris Chicaiza  

## Objetivo
Entrenar un modelo de clasificación de enfermedades foliares en zapallo (*Cucurbita moschata / pepo*) usando YOLOv11n-cls con fine-tuning desde ImageNet.

## Clases
- `healthy` — Hoja sana
- `downy_mildew` — Mildiu velloso
- `leaf_curl` — Enrollamiento foliar (virus)
- `mosaic_virus` — Virus del mosaico
- `red_beetle` — Daño por escarabajo rojo

## ⚠️ Requisito: GPU habilitada
En Colab: `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → `T4 GPU`

In [ ]:
# ── 0. Verificar GPU ─────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '⚠️ GPU no disponible. Activar GPU en Colab.')

In [ ]:
# ── 1. Instalar dependencias ─────────────────────────────────────
!pip install ultralytics albumentations -q
print('✅ Dependencias instaladas')

In [ ]:
# ── 2. Montar Google Drive (donde están los datasets) ────────────
from google.colab import drive
drive.mount('/content/drive')

# Ajustar esta ruta a donde subiste los datasets
DRIVE_BASE = '/content/drive/MyDrive/ZapalloAI'
DATA_DIR = '/content/zapallo_data'

import os
os.makedirs(DATA_DIR, exist_ok=True)
print(f'✅ Drive montado. Datos en: {DRIVE_BASE}')

In [ ]:
# ── 3. Copiar y organizar datasets ───────────────────────────────
import shutil
from pathlib import Path

# Mapeo de clases entre datasets
CLASS_MAP = {
    # Sweet Pumpkin Disease Recognition
    'Healthy': 'healthy',
    'Powdery Mildew': 'downy_mildew',
    'Leaf Curl': 'leaf_curl',
    'Mosaic Virus': 'mosaic_virus',
    'Red Beetle': 'red_beetle',
    # Cucurbit Leaf Disease Dataset (ajustar nombres según dataset real)
    'healthy': 'healthy',
    'downy_mildew': 'downy_mildew',
    'leaf_curl_virus': 'leaf_curl',
    'mosaic': 'mosaic_virus',
}

CLASSES = ['healthy', 'downy_mildew', 'leaf_curl', 'mosaic_virus', 'red_beetle']

# Crear estructura de directorios
for split in ['train', 'val', 'test']:
    for cls in CLASSES:
        os.makedirs(f'{DATA_DIR}/{split}/{cls}', exist_ok=True)

print('✅ Estructura de carpetas creada')
print(f'Clases: {CLASSES}')

In [ ]:
# ── 4. Copiar imágenes desde Drive y hacer split ─────────────────
import random
from pathlib import Path

random.seed(42)  # Reproducibilidad

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
# TEST_RATIO  = 0.15 (el resto)

total_copied = {cls: 0 for cls in CLASSES}

# ⚠️ Ajustar esta ruta al directorio real del dataset en Drive
SWEET_PUMPKIN_DIR = Path(f'{DRIVE_BASE}/sweet_pumpkin')
CUCURBIT_DIR      = Path(f'{DRIVE_BASE}/cucurbit_leaf')

def copy_dataset(source_dir: Path, class_map: dict):
    for src_cls_dir in source_dir.iterdir():
        if not src_cls_dir.is_dir():
            continue
        dst_cls = class_map.get(src_cls_dir.name)
        if dst_cls is None:
            print(f'  ⚠️ Clase no mapeada: {src_cls_dir.name}')
            continue
        
        images = list(src_cls_dir.glob('*.jpg')) + list(src_cls_dir.glob('*.png'))
        random.shuffle(images)
        
        n = len(images)
        n_train = int(n * TRAIN_RATIO)
        n_val   = int(n * VAL_RATIO)
        
        splits = {
            'train': images[:n_train],
            'val':   images[n_train:n_train+n_val],
            'test':  images[n_train+n_val:],
        }
        
        for split, imgs in splits.items():
            for img in imgs:
                dst = Path(f'{DATA_DIR}/{split}/{dst_cls}/{img.name}')
                if not dst.exists():  # Evitar sobrescribir
                    shutil.copy2(img, dst)
        
        total_copied[dst_cls] += n
        print(f'  ✅ {src_cls_dir.name} → {dst_cls}: {n} imágenes')

# Copiar ambos datasets
if SWEET_PUMPKIN_DIR.exists():
    print('📂 Sweet Pumpkin Disease Recognition:')
    copy_dataset(SWEET_PUMPKIN_DIR, CLASS_MAP)

if CUCURBIT_DIR.exists():
    print('📂 Cucurbit Leaf Disease Dataset:')
    copy_dataset(CUCURBIT_DIR, CLASS_MAP)

print('\n📊 Total por clase:', total_copied)

In [ ]:
# ── 5. Verificar distribución del dataset ─────────────────────────
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

counts = {split: {} for split in ['train', 'val', 'test']}
for split in ['train', 'val', 'test']:
    for cls in CLASSES:
        n = len(list(Path(f'{DATA_DIR}/{split}/{cls}').glob('*.jpg')))
        n += len(list(Path(f'{DATA_DIR}/{split}/{cls}').glob('*.png')))
        counts[split][cls] = n

# Mostrar tabla
print(f"{'Clase':<20} {'Train':>8} {'Val':>8} {'Test':>8} {'Total':>8}")
print('-' * 48)
for cls in CLASSES:
    t = counts['train'].get(cls, 0)
    v = counts['val'].get(cls, 0)
    ts = counts['test'].get(cls, 0)
    print(f"{cls:<20} {t:>8} {v:>8} {ts:>8} {t+v+ts:>8}")

# Gráfico de barras
fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(CLASSES))
train_counts = [counts['train'].get(c, 0) for c in CLASSES]
val_counts   = [counts['val'].get(c, 0) for c in CLASSES]
test_counts  = [counts['test'].get(c, 0) for c in CLASSES]

ax.bar(x, train_counts, label='Train', color='#2D6A4F')
ax.bar(x, val_counts, bottom=train_counts, label='Val', color='#52B788')
ax.bar(x, test_counts,
       bottom=[t+v for t,v in zip(train_counts, val_counts)],
       label='Test', color='#D8F3DC')

ax.set_xticks(list(x))
ax.set_xticklabels(CLASSES, rotation=15, ha='right')
ax.set_ylabel('Número de imágenes')
ax.set_title('Distribución del Dataset ZapalloAI')
ax.legend()
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/ZapalloAI/dataset_distribution.png', dpi=150)
plt.show()

In [ ]:
# ── 6. Entrenamiento YOLOv11n-cls ────────────────────────────────
from ultralytics import YOLO

# Cargar modelo base pre-entrenado en ImageNet
model = YOLO('yolo11n-cls.pt')

print('🚀 Iniciando entrenamiento...')
print(f'   Dataset: {DATA_DIR}')
print(f'   Clases:  {CLASSES}')

results = model.train(
    data=DATA_DIR,          # Carpeta con train/ y val/
    epochs=100,             # Máximo de épocas
    imgsz=224,              # Resolución de entrada
    batch=32,               # Batch size (T4 soporta 32 con int8)
    patience=20,            # Early stopping si no mejora
    optimizer='AdamW',
    lr0=0.001,              # Learning rate inicial
    lrf=0.01,               # LR final = lr0 * lrf
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    cos_lr=True,            # Cosine LR scheduler
    augment=True,           # Augmentation built-in de Ultralytics
    degrees=30,             # Rotación (hojas: todas las orientaciones)
    fliplr=0.5,
    flipud=0.3,
    hsv_h=0.015,            # Variación de tono (condiciones de luz)
    hsv_s=0.7,
    hsv_v=0.4,
    erasing=0.4,            # Random erasing
    mixup=0.1,
    project='/content/drive/MyDrive/ZapalloAI/runs/classify',
    name='zapallo_yolov11n_v1',
    exist_ok=True,
    device=0,               # GPU 0
    workers=2,
    verbose=True,
)

print('\n✅ Entrenamiento completado!')
print(f'   Mejor accuracy: {results.results_dict}')

In [ ]:
# ── 7. Evaluación en test set ─────────────────────────────────────
from ultralytics import YOLO
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from pathlib import Path

BEST_MODEL = '/content/drive/MyDrive/ZapalloAI/runs/classify/zapallo_yolov11n_v1/weights/best.pt'

model = YOLO(BEST_MODEL)

# Evaluación en test set
metrics = model.val(
    data=DATA_DIR,
    split='test',
    imgsz=224,
    batch=32,
    device=0,
)

print('📊 Métricas en Test Set:')
print(f'   Top-1 Accuracy: {metrics.top1:.4f} ({metrics.top1*100:.2f}%)')
print(f'   Top-5 Accuracy: {metrics.top5:.4f} ({metrics.top5*100:.2f}%)')

In [ ]:
# ── 8. Matriz de confusión ─────────────────────────────────────────
from PIL import Image
import torch

all_preds = []
all_labels = []

for cls_idx, cls_name in enumerate(CLASSES):
    cls_dir = Path(f'{DATA_DIR}/test/{cls_name}')
    images = list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png'))
    
    for img_path in images:
        result = model(str(img_path), verbose=False)[0]
        pred_cls = result.probs.top1
        all_preds.append(pred_cls)
        all_labels.append(cls_idx)

# Classification report
print('\n📋 Reporte de Clasificación:')
print(classification_report(all_labels, all_preds, target_names=CLASSES, digits=4))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0])
axes[0].set_title('Matriz de Confusión (conteos)')
axes[0].set_ylabel('Real')
axes[0].set_xlabel('Predicho')

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[1])
axes[1].set_title('Matriz de Confusión (normalizada)')
axes[1].set_ylabel('Real')
axes[1].set_xlabel('Predicho')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/ZapalloAI/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Confusion matrix guardada en Drive')

In [ ]:
# ── 9. Exportación a TFLite ────────────────────────────────────────
from ultralytics import YOLO

BEST_MODEL = '/content/drive/MyDrive/ZapalloAI/runs/classify/zapallo_yolov11n_v1/weights/best.pt'
model = YOLO(BEST_MODEL)

print('📦 Exportando a TFLite float32...')
model.export(
    format='tflite',
    imgsz=224,
    # int8=False  (float32 primero para validar accuracy)
)

print('📦 Exportando a TFLite int8 (cuantizado)...')
model.export(
    format='tflite',
    int8=True,
    imgsz=224,
    data=DATA_DIR,       # Imágenes de calibración para cuantización
)

print('\n✅ Modelos exportados!')

# Copiar a Drive
import shutil, glob
tflite_files = glob.glob('/content/**/*.tflite', recursive=True)
for f in tflite_files:
    dst = f'/content/drive/MyDrive/ZapalloAI/exports/{Path(f).name}'
    shutil.copy2(f, dst)
    size_mb = Path(f).stat().st_size / (1024*1024)
    print(f'  Copiado: {Path(f).name} ({size_mb:.2f} MB)')

In [ ]:
# ── 10. Validar accuracy post-cuantización ────────────────────────
import tensorflow as tf
import numpy as np
from PIL import Image
from pathlib import Path

def load_tflite_model(model_path: str):
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()
    return interpreter

def predict_tflite(interpreter, image_path: str):
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    img = Image.open(image_path).resize((224, 224))
    img_array = np.array(img, dtype=np.float32) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    
    interpreter.set_tensor(input_details[0]['index'], img_array)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])
    return np.argmax(output[0])

# Comparar float32 vs int8
float32_path = glob.glob('/content/drive/MyDrive/ZapalloAI/exports/*float32*.tflite')
int8_path    = glob.glob('/content/drive/MyDrive/ZapalloAI/exports/*int8*.tflite')

if float32_path and int8_path:
    interp_f32  = load_tflite_model(float32_path[0])
    interp_int8 = load_tflite_model(int8_path[0])
    
    correct_f32 = correct_int8 = total = 0
    
    for cls_idx, cls_name in enumerate(CLASSES):
        cls_dir = Path(f'{DATA_DIR}/test/{cls_name}')
        images = list(cls_dir.glob('*.jpg'))[:20]  # 20 por clase para rapidez
        
        for img_path in images:
            pred_f32  = predict_tflite(interp_f32,  str(img_path))
            pred_int8 = predict_tflite(interp_int8, str(img_path))
            correct_f32  += (pred_f32  == cls_idx)
            correct_int8 += (pred_int8 == cls_idx)
            total += 1
    
    acc_f32  = correct_f32  / total
    acc_int8 = correct_int8 / total
    drop     = (acc_f32 - acc_int8) * 100
    
    print(f'📊 Comparación post-cuantización:')
    print(f'   Float32 accuracy: {acc_f32*100:.2f}%')
    print(f'   INT8    accuracy: {acc_int8*100:.2f}%')
    print(f'   Drop accuracy:    {drop:.2f}% (objetivo < 5%)')
    
    if drop <= 3:
        print('✅ Cuantización exitosa — drop dentro del rango aceptable')
    elif drop <= 5:
        print('⚠️ Drop aceptable pero en el límite. Verificar en campo.')
    else:
        print('❌ Drop mayor al 5%. Considerar float16 en lugar de int8.')
else:
    print('⚠️ No se encontraron modelos TFLite. Ejecutar celdas de exportación.')

In [ ]:
# ── 11. Generar labels.txt para la app Flutter ────────────────────
labels_path = '/content/drive/MyDrive/ZapalloAI/exports/labels.txt'

# Labels en el mismo orden que las clases del modelo
labels = [
    'healthy',
    'downy_mildew',
    'leaf_curl',
    'mosaic_virus',
    'red_beetle',
]

with open(labels_path, 'w') as f:
    f.write('\n'.join(labels))

print('✅ labels.txt generado:')
print('\n'.join(f'  {i}: {l}' for i, l in enumerate(labels)))
print(f'\nUbicación: {labels_path}')
print('\n📱 PRÓXIMO PASO: Copiar estos archivos a la app Flutter:')
print('  → zapallo_app/assets/models/best_int8.tflite')
print('  → zapallo_app/assets/models/labels.txt')

## ✅ Resumen del Sprint 3

| Ítem | Resultado |
|---|---|
| Modelo base | YOLOv11n-cls (ImageNet) |
| Dataset total | ~11,000 imágenes |
| Épocas | Hasta 100 (early stopping) |
| Resolución | 224×224 px |
| Accuracy objetivo | ≥92% |
| Drop cuantización | <5% |
| Tamaño modelo INT8 | ~4-8 MB |

### Archivos generados
- `exports/best.tflite` — Modelo float32
- `exports/best_int8.tflite` — Modelo cuantizado (para la app)
- `exports/labels.txt` — Etiquetas en orden
- `confusion_matrix.png` — Matriz de confusión
- `dataset_distribution.png` — Distribución de clases

### Próximo: Sprint 4
Integrar `best_int8.tflite` en la app Flutter con `tflite_flutter`.